# Figure 05 -- interaction counts vs N

Loads `bench/results/scaling/interaction_counts.json`, produced by `bench/scaling/interaction_counts.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.RESULTS_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("scaling/interaction_counts.json")
cfg, recs, fits = art["config"], art["data"]["records"], art["data"]["fits"]
recs = [r for r in recs if "error" not in r]

fig, axes = style.figure(width=style.TWO_COL, height=2.7, ncols=2)

# Left: raw counts. Right: counts per particle, which is where an O(N) result
# either shows up as a flat line or does not.
for ax, per_particle in ((axes[0], False), (axes[1], True)):
    for i, (field, label) in enumerate(
        (("far_pairs", "M2L (far) pairs"), ("near_pairs", "P2P (near) pairs"))
    ):
        sel = sorted((r for r in recs if r.get(field)), key=lambda r: r["n"])
        if not sel:
            continue
        ys = [
            (r[field] / r["n"]) if per_particle else r[field] for r in sel
        ]
        fit = fits.get(field, {})
        alpha = fit.get("exponent")
        suffix = "" if per_particle or not alpha else f"  ($\\alpha={alpha:.2f}$)"
        ax.plot(
            [r["n"] for r in sel], ys,
            marker=style.MARKERS[i % len(style.MARKERS)],
            color=style.CATEGORICAL[i],
            label=label + suffix,
            markersize=3.4,
        )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("$N$")
    ax.set_ylabel("interactions per particle" if per_particle else "interaction count")
    style.finish(ax, legend_kwargs={"loc": "best", "fontsize": 6.4})

fig.tight_layout()
# A ladder point that failed is stated, not quietly dropped: silently omitting it
# would make the sweep look like it covered a range it did not reach.
failed = [r for r in art["data"]["records"] if "error" in r]
if failed:
    axes[0].text(
        0.02, 0.98,
        "not measured: N = "
        + ", ".join(f"{r['n']:,}" for r in failed)
        + "\n(out of memory)",
        transform=axes[0].transAxes, va="top", ha="left",
        fontsize=6.0, color=style.INK_MUTED,
    )
    print("not measured:", [(r["n"], r["error"][:60]) for r in failed])

style.footer(
    fig,
    jsonio.config_caption(cfg, ["order", "theta", "basis", "leaf_size", "device", "seed"])
    + f"   exponent fitted for N >= {cfg.get('fit_min_n')}",
)
style.save(fig, FIG_DIR / "fig05_interaction_counts.pdf")

for name, fit in sorted(fits.items()):
    print(f"{name:<12s} alpha={fit['exponent']:.3f} R2={fit['r_squared']:.4f} "
          f"n={fit['n_points']} over N={fit.get('fit_min_n')}..{fit.get('fit_max_n')}")


## Caption

Accepted M2L (far) and P2P (near) interaction counts against $N$ at fixed opening
angle and leaf size, with fitted power-law exponents. These are properties of the
tree and the acceptance criterion alone, independent of hardware, and so are a
cleaner statement of complexity than wall-clock (figure 4), which additionally
mixes in memory bandwidth and kernel launch overhead.

**Both exponents exceed one** -- $\alpha = 1.32$ for M2L and $1.20$ for P2P
($R^2 = 0.996$ and $0.9995$) -- and the right panel shows why: the counts *per
particle* are not constant but grow, from 2.6 to 7.7 M2L pairs per particle across
a factor of 32 in $N$. A factor of ~3 in response to a factor of 32 in $N$ is
$\log N$ growth, so the measurement is consistent with $O(N \log N)$ rather than
strictly linear, which is what a fixed leaf size and a fixed opening angle should
give: the tree deepens with $N$, and each cell acquires more well-separated
partners. The fitted single exponent is the power law that best approximates
$N \log N$ over this range, not evidence of a different asymptotic class.

Counts come from the solver's public runtime diagnostics after `prepare_state`.
Any ladder point that could not be measured is named on the figure. Values from
`bench/results/scaling/interaction_counts.json`.
